# Week 2, Day 1-4: Fine-Tuning a Transformer on CUAD Clause Classification

**Run this notebook on Kaggle with GPU enabled** (Settings → Accelerator → GPU T4 x2 or P100).

This fine-tunes `roberta-base` to classify whether a given clause category is present in a contract excerpt — same task as `train_classical_baseline.py`, but with a transformer.

**Before running:** Upload `train.jsonl` and `val.jsonl` (from `data/processed/`) as a Kaggle Dataset.

In [ ]:
!pip install -q transformers datasets scikit-learn accelerate

In [ ]:
import json
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

MODEL_NAME = "roberta-base"
MAX_LENGTH = 256

TARGET_CATEGORIES = [
    "Termination For Convenience",
    "Anti-Assignment",
    "Governing Law",
    "Cap On Liability",
    "Non-Compete",
]

In [ ]:
TRAIN_PATH = "/kaggle/input/cuad-clause-data/train.jsonl"
VAL_PATH = "/kaggle/input/cuad-clause-data/val.jsonl"

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

def build_text(row):
    return f"[CATEGORY: {row['category']}] {row['text']}"

def filter_categories(rows, categories):
    return [r for r in rows if any(c.lower() in r["category"].lower() for c in categories)]

train_rows = filter_categories(load_jsonl(TRAIN_PATH), TARGET_CATEGORIES)
val_rows = filter_categories(load_jsonl(VAL_PATH), TARGET_CATEGORIES)

print(f"Train: {len(train_rows)} | Val: {len(val_rows)}")

train_ds = Dataset.from_dict({
    "text": [build_text(r) for r in train_rows],
    "label": [r["label"] for r in train_rows],
})
val_ds = Dataset.from_dict({
    "text": [build_text(r) for r in val_rows],
    "label": [r["label"] for r in val_rows],
})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

train_ds = train_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")
train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

training_args = TrainingArguments(
    output_dir="/kaggle/working/results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
metrics = trainer.evaluate()
print(metrics)

with open("/kaggle/working/transformer_results.json", "w") as f:
    json.dump(metrics, f, indent=2)

In [ ]:
trainer.save_model("/kaggle/working/fine_tuned_clause_model")
tokenizer.save_pretrained("/kaggle/working/fine_tuned_clause_model")
print("Model saved. Download the fine_tuned_clause_model folder from Kaggle's Output panel.")